In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
import scene_generation.core as core_mod

from pathlib import Path
from scene_generation.core import Scene
from scene_generation.utils import rect_from_point_and_size, get_utm_epsg_code_from_gps, gps_to_utm_xy
from scene_generation.itu_materials import ITU_MATERIALS
from collections import Counter
from sionna.rt import scene, preview

In [ ]:
# Scene center
# DuPont Circle: -77.043446, 38.909647
# Duke Wilkinson area: -78.940297, 36.002556
CENTER_LON = -77.043446
CENTER_LAT  = 38.909647

SCENE_WIDTH  = 500   # east–west extent, metres
SCENE_HEIGHT = 500   # north–south extent, metres

DATA_DIR_LIDAR_OSM = "./scenes/dupont_lidar_osm"
DATA_DIR_OVERTURE = "./scenes/dupont_overture"

OSM_SERVER = "https://overpass-api.de/api/interpreter"

In [ ]:
for out_dir in (DATA_DIR_LIDAR_OSM, DATA_DIR_OVERTURE):
    os.makedirs(out_dir, exist_ok=True)
    print(f"Output directory: {os.path.abspath(out_dir)}")

In [ ]:
def generate_scene(mode, out_dir, *, track_height_sources=False):
    """Generate the scene and optionally count selected height sources."""
    scene_polygon = rect_from_point_and_size(
    CENTER_LON, CENTER_LAT, "center", SCENE_WIDTH, SCENE_HEIGHT
    )

    height_sources = []
    original_resolve = core_mod.resolve_building_height

    def logging_resolve_building_height(*args, **kwargs):
        kwargs = dict(kwargs)
        kwargs["return_source"] = True
        height, metadata = original_resolve(*args, **kwargs)
        height_sources.append(metadata.get("source", "unknown"))
        return height

    if track_height_sources:
        core_mod.resolve_building_height = logging_resolve_building_height

    try:
        scene = Scene()
        building_height_map = scene(
            points=scene_polygon,
            data_dir=out_dir,
            osm_server_addr=OSM_SERVER,
            hag_tiff_path=None,  # None lets lidar-osm create/reuse out_dir/test_hag.tif
            ground_material_type="mat-itu_wet_ground",
            rooftop_material_type="mat-itu_metal",
            wall_material_type="mat-itu_concrete",
            generate_building_map=True,
            building_height_mode=mode,
            # lidar_terrain=False,
            # dem_terrain=False,
        )
    finally:
        if track_height_sources:
            core_mod.resolve_building_height = original_resolve

    return building_height_map, Counter(height_sources)

lidar_height_map, lidar_height_sources = generate_scene(
    "lidar-osm", DATA_DIR_LIDAR_OSM, track_height_sources=True
)
overture_height_map, overture_height_sources = generate_scene(
    "overture", DATA_DIR_OVERTURE, track_height_sources=True
)

print("Scene generation complete!")
print("LiDAR/OSM height sources:", lidar_height_sources)
print("Overture height sources :", overture_height_sources)
print("LiDAR HAG raster exists :", (Path(DATA_DIR_LIDAR_OSM) / "test_hag.tif").exists())

In [ ]:
# Compare final building-height maps from the two modes.
lidar_map = np.load(Path(DATA_DIR_LIDAR_OSM) / "2D_Building_Height_Map.npy")
overture_map = np.load(Path(DATA_DIR_OVERTURE) / "2D_Building_Height_Map.npy")

diff = lidar_map.astype(float) - overture_map.astype(float)
mean_abs_diff = np.mean(np.abs(diff[building_mask])) if building_mask.any() else 0.0
max_abs_diff = np.max(np.abs(diff)) if diff.size else 0.0

comparison = pd.DataFrame(
    [
        ("same raster", np.array_equal(lidar_map, overture_map)),
        ("mean abs diff on building pixels", round(float(mean_abs_diff), 3)),
        ("max abs diff", round(float(max_abs_diff), 3)),
    ],
    columns=["check", "value"],
)
display(comparison)

height_vmax = max(float(lidar_map.max()), float(overture_map.max()), 1.0)
diff_abs_max = max(float(np.max(np.abs(diff))), 1.0) if diff.size else 1.0

fig, axs = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
height_im = axs[0].imshow(lidar_map, cmap="viridis", vmin=0, vmax=height_vmax)
axs[0].set_title("lidar-osm")
axs[1].imshow(overture_map, cmap="viridis", vmin=0, vmax=height_vmax)
axs[1].set_title("overture")
diff_im = axs[2].imshow(diff, cmap="coolwarm", vmin=-diff_abs_max, vmax=diff_abs_max)
axs[2].set_title("lidar-osm minus overture")
height_cbar = fig.colorbar(height_im, ax=axs[:2], fraction=0.046, pad=0.04)
height_cbar.set_label("Building height (m)")
diff_cbar = fig.colorbar(diff_im, ax=axs[2], fraction=0.046, pad=0.04)
diff_cbar.set_label("Height difference (m)")
for ax in axs:
    ax.axis("off")
plt.show()

In [ ]:

lidar_osm_sionna_scene = scene.load_scene(DATA_DIR_LIDAR_OSM + "/scene.xml")
lidar_osm_sionna_scene.preview()

In [ ]:
overture_sionna_scene = scene.load_scene(DATA_DIR_OVERTURE + "/scene.xml")
overture_sionna_scene.preview()